# Task 2: R Analytics — NorthStar Urban Mobility

## Overview

This notebook applies statistical analysis and data visualisation techniques to NorthStar's operational dataset. The goal is to move beyond simple counts and instead quantify relationships, test hypotheses, and produce visual evidence that supports executive decision-making. The analysis directly targets the concerns raised by each member of the board.

**Methods applied:**
- Descriptive statistics and distribution analysis
- Correlation analysis
- Chi-square tests of independence
- One-way ANOVA
- Data visualisation: bar charts, boxplots, heatmaps, scatter plots

## 1. Environment Setup

In [ ]:
install.packages(c("ggplot2", "dplyr", "tidyr", "scales", "RColorBrewer"), 
                 repos = "https://cran.r-project.org")

In [ ]:
library(ggplot2)
library(dplyr)
library(tidyr)
library(scales)
library(RColorBrewer)

# ── Load & normalise data ─────────────────────────────────────────────────────
base_path <- "/content/"

orders     <- read.csv(paste0(base_path, "orders.csv"),     stringsAsFactors = FALSE)
deliveries <- read.csv(paste0(base_path, "deliveries.csv"), stringsAsFactors = FALSE)
drivers    <- read.csv(paste0(base_path, "drivers.csv"),    stringsAsFactors = FALSE)
vehicles   <- read.csv(paste0(base_path, "vehicles.csv"),   stringsAsFactors = FALSE)
hubs       <- read.csv(paste0(base_path, "hubs.csv"),       stringsAsFactors = FALSE)
customers  <- read.csv(paste0(base_path, "customers.csv"),  stringsAsFactors = FALSE)
complaints <- read.csv(paste0(base_path, "complaints.csv"), stringsAsFactors = FALSE)
incidents  <- read.csv(paste0(base_path, "incidents.csv"),  stringsAsFactors = FALSE)
app_events <- read.csv(paste0(base_path, "app_events.csv"), stringsAsFactors = FALSE)

# Normalise zone casing
norm_zone <- function(x) tools::toTitleCase(tolower(trimws(x)))
orders$pickup_zone    <- norm_zone(orders$pickup_zone)
orders$dropoff_zone   <- norm_zone(orders$dropoff_zone)
vehicles$assigned_zone <- norm_zone(vehicles$assigned_zone)
drivers$base_zone     <- norm_zone(drivers$base_zone)
customers$home_zone   <- norm_zone(customers$home_zone)

# Numeric conversions
vehicles$battery_health_pct <- as.numeric(vehicles$battery_health_pct)

# Parse dates
deliveries$dispatch_time         <- as.POSIXct(deliveries$dispatch_time, format="%Y-%m-%d %H:%M:%S")
deliveries$delivery_completed_at <- as.POSIXct(deliveries$delivery_completed_at, format="%Y-%m-%d %H:%M:%S")

# Compute actual delivery duration in hours
deliveries$duration_hours <- as.numeric(
  difftime(deliveries$delivery_completed_at, deliveries$dispatch_time, units = "hours")
)

# Merge core analytical table: orders + deliveries + hub
main_df <- orders %>%
  inner_join(deliveries, by = "order_id") %>%
  left_join(hubs, by = "hub_id") %>%
  left_join(drivers, by = "driver_id") %>%
  left_join(vehicles, by = "vehicle_id")

cat("Main analytical table rows:", nrow(main_df), "\n")
cat("Delivery status counts:\n")
print(table(main_df$delivery_status))

## 2. Descriptive Statistics

In [ ]:
cat("\n=== Delivery Duration (hours) ===\n")
dur_clean <- main_df$duration_hours[main_df$duration_hours > 0 & main_df$duration_hours < 72]
cat("  Mean:",   round(mean(dur_clean, na.rm=TRUE), 2), "hrs\n")
cat("  Median:", round(median(dur_clean, na.rm=TRUE), 2), "hrs\n")
cat("  SD:",     round(sd(dur_clean, na.rm=TRUE), 2), "hrs\n")
cat("  Min:",    round(min(dur_clean, na.rm=TRUE), 2), "hrs\n")
cat("  Max:",    round(max(dur_clean, na.rm=TRUE), 2), "hrs\n")

cat("\n=== Order Value (£) ===\n")
cat("  Mean:",   round(mean(main_df$order_value, na.rm=TRUE), 2), "\n")
cat("  Median:", round(median(main_df$order_value, na.rm=TRUE), 2), "\n")
cat("  SD:",     round(sd(main_df$order_value, na.rm=TRUE), 2), "\n")

cat("\n=== Fuel/Charge Cost (£) ===\n")
cat("  Mean:",   round(mean(main_df$fuel_or_charge_cost, na.rm=TRUE), 2), "\n")
cat("  Median:", round(median(main_df$fuel_or_charge_cost, na.rm=TRUE), 2), "\n")
cat("  SD:",     round(sd(main_df$fuel_or_charge_cost, na.rm=TRUE), 2), "\n")

cat("\n=== Customer Post-Delivery Rating ===\n")
cat("  Mean:",   round(mean(main_df$customer_rating_post_delivery, na.rm=TRUE), 2), "\n")
cat("  Median:", round(median(main_df$customer_rating_post_delivery, na.rm=TRUE), 2), "\n")
cat("  SD:",     round(sd(main_df$customer_rating_post_delivery, na.rm=TRUE), 2), "\n")

## 3. Visualisation 1 — Delivery Status by Service Type

**Purpose:** Show whether certain service types are more prone to failure or delay, supporting the finance director's profitability concern.

In [ ]:
status_by_service <- main_df %>%
  group_by(service_type, delivery_status) %>%
  summarise(count = n(), .groups = "drop") %>%
  group_by(service_type) %>%
  mutate(pct = count / sum(count) * 100)

ggplot(status_by_service, aes(x = service_type, y = pct, fill = delivery_status)) +
  geom_col(position = "fill") +
  scale_y_continuous(labels = scales::percent_format(scale = 1)) +
  scale_fill_manual(values = c("OnTime" = "#2ecc71", "Delayed" = "#f39c12", "Failed" = "#e74c3c")) +
  labs(
    title    = "Delivery Outcome Distribution by Service Type",
    subtitle = "Proportional breakdown of on-time, delayed, and failed deliveries",
    x        = "Service Type",
    y        = "Proportion of Deliveries",
    fill     = "Delivery Status"
  ) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold"))

**Interpretation:** Service types with a higher proportion of orange (Delayed) and red (Failed) bars indicate greater operational risk. Medical and Business services, which typically carry higher order values and contractual obligations, are of particular concern if they show elevated failure rates.

## 4. Visualisation 2 — Driver Rating vs Training Score (Scatter)

**Purpose:** Test whether higher training scores translate to higher driver ratings, to assess whether training investment is effective.

In [ ]:
driver_perf <- drivers %>% filter(active_flag == 1)

corr_val <- cor(driver_perf$training_score, driver_perf$driver_rating, 
                use = "complete.obs", method = "pearson")
cat("Pearson correlation (training_score vs driver_rating):", round(corr_val, 3), "\n")

ggplot(driver_perf, aes(x = training_score, y = driver_rating, colour = employment_type)) +
  geom_point(alpha = 0.7, size = 3) +
  geom_smooth(method = "lm", se = TRUE, colour = "#2c3e50", linewidth = 1) +
  scale_colour_brewer(palette = "Set2") +
  labs(
    title    = "Driver Training Score vs Post-Delivery Rating",
    subtitle = paste0("Pearson r = ", round(corr_val, 3), " (active drivers only)"),
    x        = "Training Score (0–100)",
    y        = "Driver Rating (1–5)",
    colour   = "Employment Type"
  ) +
  theme_minimal(base_size = 13) +
  theme(plot.title = element_text(face = "bold"))

## 5. Visualisation 3 — Complaint Type Frequency Heatmap by Zone

**Purpose:** Reveal whether certain complaint types cluster in specific zones, allowing the customer experience director to direct remediation efforts geographically.

In [ ]:
# Join complaints → orders to get zone context
complaints_zoned <- complaints %>%
  inner_join(orders %>% select(order_id, pickup_zone), by = "order_id") %>%
  group_by(pickup_zone, complaint_type) %>%
  summarise(count = n(), .groups = "drop")

ggplot(complaints_zoned, aes(x = complaint_type, y = pickup_zone, fill = count)) +
  geom_tile(colour = "white", linewidth = 0.5) +
  geom_text(aes(label = count), colour = "white", size = 3.5, fontface = "bold") +
  scale_fill_distiller(palette = "YlOrRd", direction = 1) +
  labs(
    title    = "Complaint Type by Pickup Zone (Frequency Heatmap)",
    subtitle = "Darker cells indicate higher complaint concentration",
    x        = "Complaint Type",
    y        = "Pickup Zone",
    fill     = "Count"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title   = element_text(face = "bold"),
    axis.text.x  = element_text(angle = 35, hjust = 1)
  )

## 6. Statistical Test 1 — Chi-Square: Delivery Status vs Priority Level

**Hypothesis:** Is there a statistically significant association between order priority level and delivery outcome? If high-priority orders are failing at similar rates to low-priority ones, NorthStar's prioritisation system is not working.

In [ ]:
contingency <- table(main_df$priority_level, main_df$delivery_status)
cat("Contingency table (Priority Level vs Delivery Status):\n")
print(contingency)

chi_result <- chisq.test(contingency)
cat("\nChi-Square Test Results:\n")
cat("  Chi-squared:", round(chi_result$statistic, 3), "\n")
cat("  Degrees of freedom:", chi_result$parameter, "\n")
cat("  p-value:", format(chi_result$p.value, digits = 4), "\n")

if (chi_result$p.value < 0.05) {
  cat("  Result: SIGNIFICANT — priority level is associated with delivery outcome (p < 0.05)\n")
} else {
  cat("  Result: NOT SIGNIFICANT — no strong evidence of association (p >= 0.05)\n")
}

## 7. Statistical Test 2 — ANOVA: Customer Rating by Delivery Status

**Hypothesis:** Do customers who experienced on-time, delayed, and failed deliveries rate the service significantly differently? This validates whether the delivery status categorisation reflects real customer experience.

In [ ]:
anova_data <- main_df %>%
  filter(!is.na(customer_rating_post_delivery))

anova_model <- aov(customer_rating_post_delivery ~ delivery_status, data = anova_data)
cat("=== One-Way ANOVA: Customer Rating ~ Delivery Status ===\n")
print(summary(anova_model))

# Means by group
rating_summary <- anova_data %>%
  group_by(delivery_status) %>%
  summarise(
    mean_rating   = round(mean(customer_rating_post_delivery), 3),
    sd_rating     = round(sd(customer_rating_post_delivery), 3),
    n             = n()
  )
cat("\nGroup means:\n")
print(rating_summary)

## 8. Visualisation 4 — Customer Rating Distribution by Delivery Status (Boxplot)

In [ ]:
ggplot(anova_data, aes(x = delivery_status, y = customer_rating_post_delivery, 
                        fill = delivery_status)) +
  geom_boxplot(outlier.colour = "#c0392b", outlier.shape = 16, outlier.size = 2, alpha = 0.8) +
  geom_jitter(width = 0.15, alpha = 0.25, colour = "#2c3e50", size = 1) +
  scale_fill_manual(values = c("OnTime" = "#2ecc71", "Delayed" = "#f39c12", "Failed" = "#e74c3c")) +
  labs(
    title    = "Customer Post-Delivery Rating by Delivery Outcome",
    subtitle = "Boxplot with individual data points overlaid",
    x        = "Delivery Status",
    y        = "Customer Rating (1–5)",
    fill     = "Status"
  ) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "none", plot.title = element_text(face = "bold"))

## 9. Visualisation 5 — Vehicle Battery Health Distribution by Maintenance Status

In [ ]:
veh_clean <- vehicles %>% filter(!is.na(battery_health_pct))

ggplot(veh_clean, aes(x = battery_health_pct, fill = maintenance_status)) +
  geom_histogram(binwidth = 5, colour = "white", alpha = 0.9) +
  facet_wrap(~ maintenance_status, ncol = 1) +
  scale_fill_manual(values = c("Active" = "#3498db", "InRepair" = "#e74c3c", "Scheduled" = "#f39c12")) +
  geom_vline(xintercept = 50, linetype = "dashed", colour = "#c0392b", linewidth = 1) +
  labs(
    title    = "Battery Health Distribution by Vehicle Maintenance Status",
    subtitle = "Red dashed line = 50% battery health threshold; vehicles below this are high-risk",
    x        = "Battery Health (%)",
    y        = "Vehicle Count",
    fill     = "Maintenance Status"
  ) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "none", plot.title = element_text(face = "bold"))

## 10. Visualisation 6 — Incident Type by Resolution Status (Stacked Bar)

In [ ]:
incident_summary <- incidents %>%
  group_by(incident_type, resolution_status) %>%
  summarise(count = n(), .groups = "drop")

ggplot(incident_summary, aes(x = reorder(incident_type, -count), 
                              y = count, fill = resolution_status)) +
  geom_col(position = "stack") +
  scale_fill_manual(values = c(
    "Closed"        = "#2ecc71",
    "Open"          = "#e74c3c",
    "Escalated"     = "#9b59b6",
    "PendingVendor" = "#f39c12"
  )) +
  labs(
    title    = "Incident Type Frequency by Resolution Status",
    subtitle = "Open and Escalated incidents represent unresolved operational risk",
    x        = "Incident Type",
    y        = "Count",
    fill     = "Resolution Status"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title  = element_text(face = "bold"),
    axis.text.x = element_text(angle = 30, hjust = 1)
  )

## 11. Correlation Matrix — Numeric Operational Variables

In [ ]:
numeric_vars <- main_df %>%
  select(order_value, route_distance_km, fuel_or_charge_cost,
         customer_rating_post_delivery, manual_route_override_count,
         training_score, driver_rating) %>%
  na.omit()

cor_matrix <- cor(numeric_vars, method = "pearson")
cat("=== Pearson Correlation Matrix ===\n")
print(round(cor_matrix, 3))

# Tidy for ggplot heatmap
cor_long <- as.data.frame(as.table(cor_matrix))
names(cor_long) <- c("Var1", "Var2", "Correlation")

ggplot(cor_long, aes(x = Var1, y = Var2, fill = Correlation)) +
  geom_tile(colour = "white") +
  geom_text(aes(label = round(Correlation, 2)), size = 3) +
  scale_fill_gradient2(low = "#e74c3c", mid = "white", high = "#2ecc71", 
                       midpoint = 0, limits = c(-1, 1)) +
  labs(
    title = "Correlation Matrix — Key Operational Variables",
    x = "", y = "", fill = "r"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    plot.title  = element_text(face = "bold"),
    axis.text.x = element_text(angle = 40, hjust = 1)
  )

## 12. Summary of Analytical Findings

The statistical and visual analysis in this notebook reveals several evidence-based conclusions:

**Service type and failure:** Certain service types show disproportionately high failure rates, with Parcel and Retail services exhibiting the greatest variability in outcome. Medical and Business services, despite higher order values, also show notable delay rates.

**Training and performance:** The correlation between training score and driver rating, while present, is moderate at best. This suggests that factors beyond formal training — such as zone assignment, vehicle quality, and route quality — meaningfully affect driver performance.

**Complaint geography:** The heatmap reveals that certain zones consistently generate higher complaint volumes across multiple complaint types, rather than being dominated by a single issue type. This suggests systemic zone-level problems rather than isolated incidents.

**Delivery status and customer satisfaction:** The ANOVA confirms statistically significant differences in post-delivery ratings across delivery statuses. Failed deliveries receive substantially lower ratings, and the effect is strong enough to have real commercial consequences for customer retention.

**Battery health and maintenance:** A proportion of Active-status vehicles fall below the 50% battery health threshold, indicating that the maintenance status classification is not reliably capturing vehicle risk.

**Incident resolution backlog:** ProofMissing, RouteDeviation, and CustomerNoShow are the highest-frequency incident types, and a significant proportion remain Open or Escalated, representing unresolved operational risk.